In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import ds_single, ds_pressure

In [3]:
from tqdm import tqdm

In [4]:
expert = pd.read_csv("expert.csv")

In [5]:
def parse_float(x: str):
    if len(x.strip()) == 0:
        return np.nan
    return float(x.replace(",", "."))

In [6]:
from dataclasses import dataclass


@dataclass
class ExpertTolerance:
    abs_l1: float
    abs_linf: float
    rel_l1: float
    rel_linf: float

In [7]:
single_tolerances = dict()
    
for key in tqdm(sorted(ds_single.mean)):
    # patch up expert variable name mismatches
    ekey = {
        'd2m': "2d",
        'fg10': "10fg",
        'siconc': "ci",
        't2m': "2t",
        'u10': "10u",
        'u100': "100u",
        'v10': "10v",
        'v100': "100v",
    }.get(key, key)
    ex = expert[
        (expert["Short Name"] == ekey) & (expert["Level"] == "Single/Surface")
    ]
    if len(ex) > 0:
        assert len(ex) == 1
        ex = ex.iloc[0]
        absl1 = parse_float(str(ex["Absolute L1 error"]))
        abslinf = parse_float(str(ex["Absolute Linfty error"]))
        rell1 = parse_float(str(ex["Relative L1 error"]))
        rellinf = parse_float(str(ex["Relative Linfty error"]))
    else:
        absl1 = np.nan
        abslinf = np.nan
        rell1 = np.nan
        rellinf = np.nan
    single_tolerances[key] = ExpertTolerance(absl1, abslinf, rell1, rellinf)

100%|██████████████████████████████████████████████████████| 226/226 [00:00<00:00, 6466.20it/s]


In [8]:
from sklearn import tree

In [9]:
long_names = { key: f"{key} {ds_single.mean[key].long_name} {ds_single.mean[key].GRIB_stepType} {ds_single.mean[key].units}" for key in single_tolerances.keys() }

In [10]:
import sklearn
cnt = sklearn.feature_extraction.text.CountVectorizer()

In [11]:
X = cnt.fit_transform(list(long_names.values()))

In [12]:
set([str(single_tolerances[key].abs_linf) for key in single_tolerances.keys()])

{'0.01', '0.05', '0.5', '1.0', '10.0', 'nan'}

In [13]:
from matplotlib import pyplot as plt

y = [str(np.fmin(single_tolerances[key].abs_l1, single_tolerances[key].abs_linf)) for key in single_tolerances.keys()]

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 20))

sklearn.tree.plot_tree(clf, ax=ax, feature_names=[v for v, i in sorted(cnt.vocabulary_.items(), key=lambda a: a[1])], impurity=False, class_names=clf.classes_, filled=True)

fig.savefig("expert-tree-abs.pdf")

plt.close(fig)

In [14]:
from matplotlib import pyplot as plt

y = [str(single_tolerances[key].abs_l1) for key in single_tolerances.keys()]

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 20))

sklearn.tree.plot_tree(clf, ax=ax, feature_names=[v for v, i in sorted(cnt.vocabulary_.items(), key=lambda a: a[1])], impurity=False, class_names=clf.classes_)

fig.savefig("expert-tree-abs-l1.pdf")

plt.close(fig)

In [15]:
from matplotlib import pyplot as plt

y = [str(single_tolerances[key].abs_linf) for key in single_tolerances.keys()]

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 20))

sklearn.tree.plot_tree(clf, ax=ax, feature_names=[v for v, i in sorted(cnt.vocabulary_.items(), key=lambda a: a[1])], impurity=False, class_names=clf.classes_)

fig.savefig("expert-tree-abs-linf.pdf")

plt.close(fig)

In [16]:
from matplotlib import pyplot as plt

y = [str(single_tolerances[key].rel_l1) for key in single_tolerances.keys()]

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 20))

sklearn.tree.plot_tree(clf, ax=ax, feature_names=[v for v, i in sorted(cnt.vocabulary_.items(), key=lambda a: a[1])], impurity=False, class_names=clf.classes_)

fig.savefig("expert-tree-rel-l1.pdf")

plt.close(fig)

In [17]:
from matplotlib import pyplot as plt

y = [str(single_tolerances[key].rel_linf) for key in single_tolerances.keys()]

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, y)

fig, ax = plt.subplots(figsize=(20, 20))

sklearn.tree.plot_tree(clf, ax=ax, feature_names=[v for v, i in sorted(cnt.vocabulary_.items(), key=lambda a: a[1])], impurity=False, class_names=clf.classes_)

fig.savefig("expert-tree-rel-linf.pdf")

plt.close(fig)